# TF-IDF + Classical ML for BI-RADS Classification

Build TF-IDF feature extraction (word + char n-grams) combined with regex features.  
Train LightGBM, XGBoost, and CatBoost with class-weight balancing and Optuna HPO.  
Compare against the regex baseline (US1) using macro-F1.

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 5
METRIC = "f1_macro"
DATA_PATH = "data/raw/train.csv"
MODEL_TYPE = "lightgbm"
N_TRIALS = 20
MAX_WORD_FEATURES = 10000
MAX_CHAR_FEATURES = 20000
MLFLOW_EXPERIMENT = "birads-tfidf"

In [ ]:
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from scipy import sparse
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

matplotlib.use("Agg")
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

COMPETITION = "spr-2026-mammography-report-classification"

KAGGLE_INPUT = Path("/kaggle/input") / COMPETITION
if KAGGLE_INPUT.exists():
    DATA_PATH = str(KAGGLE_INPUT / "train.csv")
    IS_KAGGLE = True
else:
    IS_KAGGLE = False


def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    return Path.cwd()


ROOT = _find_project_root()
_data_path = Path(DATA_PATH)
if not _data_path.is_absolute():
    _data_path = ROOT / _data_path
DATA_PATH = str(_data_path)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"Project root: {ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Kaggle env: {IS_KAGGLE}")

## Load Data & Extract Regex Features

In [ ]:
from preprocessing.preprocess import (
    ACHADOS_PATTERNS,
    build_tfidf_pipeline,
    extract_features,
)

train_df = pd.read_csv(DATA_PATH)
print(f"Train shape: {train_df.shape}")
print(f"\nTarget distribution:")
print(train_df["target"].value_counts().sort_index())

train_df = extract_features(train_df)

y = train_df["target"].values
class_labels = sorted(np.unique(y))
print(f"\nClasses: {class_labels}")

## Build TF-IDF + Regex Feature Pipeline

In [ ]:
achado_cols = list(ACHADOS_PATTERNS.keys())
tfidf_input_cols = ["report", "indicacao_class"] + achado_cols

X = train_df[tfidf_input_cols].copy()

pipeline = build_tfidf_pipeline(
    max_word_features=MAX_WORD_FEATURES,
    max_char_features=MAX_CHAR_FEATURES,
)

print(f"Pipeline transformers: {[t[0] for t in pipeline.transformers]}")
print(f"Input columns: {tfidf_input_cols}")

## Model Definitions

In [ ]:
n_classes = len(class_labels)
class_counts = pd.Series(y).value_counts().sort_index()
total = len(y)
catboost_class_weights = {
    int(c): total / (n_classes * count) for c, count in class_counts.items()
}


def build_model_and_params(model_type, trial):
    """Return (model, params_dict) for a given model type and Optuna trial."""
    if model_type == "lightgbm":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        model = LGBMClassifier(
            class_weight="balanced",
            verbosity=-1,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            **params,
        )
    elif model_type == "xgboost":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        }
        sample_weight_ratio = {
            c: total / (n_classes * cnt) for c, cnt in class_counts.items()
        }
        scale = sample_weight_ratio.get(class_labels[0], 1.0)
        model = XGBClassifier(
            scale_pos_weight=scale,
            verbosity=0,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            use_label_encoder=False,
            eval_metric="mlogloss",
            **params,
        )
    elif model_type == "catboost":
        params = {
            "iterations": trial.suggest_int("iterations", 200, 1000, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "depth": trial.suggest_int("depth", 3, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
            "bagging_temperature": trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            ),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-2, 10.0, log=True
            ),
        }
        model = CatBoostClassifier(
            class_weights=catboost_class_weights,
            verbose=0,
            random_state=RANDOM_STATE,
            **params,
        )
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return model, params

## Optuna HPO + Stratified K-Fold CV Training

In [ ]:
MODEL_TYPES = ["lightgbm", "xgboost", "catboost"]

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Precompute TF-IDF features per fold once (the expensive step)
print("Precomputing TF-IDF features per fold...")
fold_data = []
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    pipe = build_tfidf_pipeline(
        max_word_features=MAX_WORD_FEATURES,
        max_char_features=MAX_CHAR_FEATURES,
    )
    X_tr = pipe.fit_transform(X.iloc[train_idx])
    X_va = pipe.transform(X.iloc[val_idx])

    if sparse.issparse(X_tr):
        X_tr = X_tr.toarray()
        X_va = X_va.toarray()

    fold_data.append(
        {
            "X_tr": X_tr,
            "X_va": X_va,
            "y_tr": y[train_idx],
            "y_va": y[val_idx],
            "val_idx": val_idx,
        }
    )
    print(f"  Fold {fold_idx + 1}/{N_FOLDS}: train {X_tr.shape}, val {X_va.shape}")

print("TF-IDF precomputation done.\n")

all_results = {}

for mtype in MODEL_TYPES:
    print(f"\n{'='*60}")
    print(f"  Training: {mtype}  ({N_TRIALS} Optuna trials × {N_FOLDS}-fold CV)")
    print(f"{'='*60}")

    def objective(trial, _mtype=mtype):
        fold_scores = []
        for fd in fold_data:
            model, _ = build_model_and_params(_mtype, trial)
            model.fit(fd["X_tr"], fd["y_tr"])
            y_pred = model.predict(fd["X_va"])
            fold_scores.append(
                f1_score(fd["y_va"], y_pred, average="macro", zero_division=0)
            )
        return np.mean(fold_scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"  Best CV macro-F1: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

    oof_preds = np.zeros(len(y), dtype=int)
    fold_f1_scores = []
    fold_reports = []

    for fold_idx, fd in enumerate(fold_data):
        best_trial = optuna.trial.FixedTrial(study.best_params)
        model, _ = build_model_and_params(mtype, best_trial)
        model.fit(fd["X_tr"], fd["y_tr"])
        y_pred = model.predict(fd["X_va"])
        oof_preds[fd["val_idx"]] = y_pred

        fold_f1 = f1_score(fd["y_va"], y_pred, average="macro", zero_division=0)
        fold_f1_scores.append(fold_f1)
        fold_reports.append(
            classification_report(fd["y_va"], y_pred, output_dict=True, zero_division=0)
        )
        print(f"  Fold {fold_idx + 1}/{N_FOLDS}: macro-F1 = {fold_f1:.4f}")

    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    print(f"  Mean macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")

    all_results[mtype] = {
        "study": study,
        "best_params": study.best_params,
        "fold_f1_scores": fold_f1_scores,
        "fold_reports": fold_reports,
        "mean_f1": mean_f1,
        "std_f1": std_f1,
        "oof_preds": oof_preds.copy(),
    }

## Evaluation & Visualization

In [ ]:
# Per-class metrics for the best model
summary_rows = []
for mtype, res in all_results.items():
    summary_rows.append(
        {
            "Model": mtype,
            "Feature Set": "v2.0 (tfidf+regex)",
            "macro-F1 (mean)": res["mean_f1"],
            "macro-F1 (std)": res["std_f1"],
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("macro-F1 (mean)", ascending=False)
print(summary_df.to_string(index=False))

In [ ]:
best_model_name = summary_df.iloc[0]["Model"]
best_res = all_results[best_model_name]

print(f"\nBest model: {best_model_name}")
print(f"\nPer-class metrics (OOF):")
print(classification_report(y, best_res["oof_preds"], zero_division=0))

In [ ]:
# Confusion matrix for each model
n_models = len(all_results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]

for ax, (mtype, res) in zip(axes, all_results.items()):
    ConfusionMatrixDisplay.from_predictions(
        y, res["oof_preds"], ax=ax, labels=class_labels
    )
    ax.set_title(f"{mtype} — OOF macro-F1 = {res['mean_f1']:.4f}")

plt.tight_layout()
plt.show()

In [ ]:
# Comparison bar chart: all models vs regex baseline
REGEX_BASELINE_F1 = None  # will be filled from MLflow or manually

model_names = list(all_results.keys())
model_f1s = [all_results[m]["mean_f1"] for m in model_names]

if REGEX_BASELINE_F1 is not None:
    model_names = ["Regex+LR baseline"] + model_names
    model_f1s = [REGEX_BASELINE_F1] + model_f1s

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))
bars = ax.barh(model_names, model_f1s, color=colors)
ax.set_xlabel("macro-F1 (CV mean)")
ax.set_title("Model Comparison — macro-F1")
ax.set_xlim(0, max(model_f1s) * 1.2)

for bar, val in zip(bars, model_f1s):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 breakdown across models
per_class_data = []
for mtype, res in all_results.items():
    oof_report = classification_report(
        y, res["oof_preds"], output_dict=True, zero_division=0
    )
    for label in class_labels:
        label_str = str(label)
        if label_str in oof_report:
            per_class_data.append(
                {
                    "Model": mtype,
                    "Class": label,
                    "F1": oof_report[label_str]["f1-score"],
                    "Precision": oof_report[label_str]["precision"],
                    "Recall": oof_report[label_str]["recall"],
                    "Support": oof_report[label_str]["support"],
                }
            )

per_class_df = pd.DataFrame(per_class_data)
for mtype in all_results:
    print(f"\n--- {mtype} ---")
    subset = per_class_df[per_class_df["Model"] == mtype]
    print(
        subset[["Class", "Precision", "Recall", "F1", "Support"]].to_string(index=False)
    )

## Log All Runs to MLflow

In [ ]:
if not IS_KAGGLE:
    import mlflow
    from dotenv import load_dotenv
    import os

    load_dotenv(ROOT / ".env", override=True)
    for token_var in ("AWS_SESSION_TOKEN", "AWS_SECURITY_TOKEN"):
        os.environ.pop(token_var, None)

    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    for mtype, res in all_results.items():
        oof_report = classification_report(
            y, res["oof_preds"], output_dict=True, zero_division=0
        )

        with mlflow.start_run(run_name=f"tfidf-{mtype}"):
            mlflow.set_tag("model_type", mtype)
            mlflow.set_tag("feature_set_version", "v2.0")
            mlflow.set_tag("cv_strategy", f"stratified_{N_FOLDS}fold")
            mlflow.set_tag("baseline", "false")

            mlflow.log_metric("cv_f1_macro", res["mean_f1"])
            mlflow.log_metric("cv_f1_macro_std", res["std_f1"])

            for fold_i, f1_val in enumerate(res["fold_f1_scores"]):
                mlflow.log_metric(f"f1_macro_fold_{fold_i}", f1_val)

            for label in class_labels:
                label_str = str(label)
                if label_str in oof_report:
                    mlflow.log_metric(
                        f"f1_class_{label}", oof_report[label_str]["f1-score"]
                    )
                    mlflow.log_metric(
                        f"precision_class_{label}", oof_report[label_str]["precision"]
                    )
                    mlflow.log_metric(
                        f"recall_class_{label}", oof_report[label_str]["recall"]
                    )

            mlflow.log_params(res["best_params"])
            mlflow.log_param("n_trials", N_TRIALS)
            mlflow.log_param("n_folds", N_FOLDS)
            mlflow.log_param("max_word_features", MAX_WORD_FEATURES)
            mlflow.log_param("max_char_features", MAX_CHAR_FEATURES)

            with tempfile.TemporaryDirectory() as tmpdir:
                fig, ax = plt.subplots(figsize=(8, 6))
                ConfusionMatrixDisplay.from_predictions(
                    y, res["oof_preds"], ax=ax, labels=class_labels
                )
                ax.set_title(f"{mtype} — OOF Confusion Matrix")
                cm_path = Path(tmpdir) / "confusion_matrix.png"
                fig.savefig(cm_path, dpi=150, bbox_inches="tight")
                plt.close(fig)
                mlflow.log_artifact(str(cm_path), artifact_path="plots")

        print(f"{mtype} logged to MLflow.")

    print("\nAll models logged to MLflow.")
else:
    print("Kaggle environment detected — skipping MLflow logging.")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("  PHASE 4 RESULTS SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

best_f1 = summary_df.iloc[0]["macro-F1 (mean)"]
print(f"\nBest model: {best_model_name} with macro-F1 = {best_f1:.4f}")

if best_f1 > 0.50:
    print("\n✓ Gate PASSED: macro-F1 > 0.50 — proceed to Phase 5 (Transformer).")
else:
    print(
        "\n✗ Gate FAILED: macro-F1 ≤ 0.50 — investigate feature engineering before transformers."
    )

# Check minority class recall
minority_classes = [4, 5, 6]
for mc in minority_classes:
    mc_data = per_class_df[
        (per_class_df["Model"] == best_model_name) & (per_class_df["Class"] == mc)
    ]
    if not mc_data.empty:
        recall = mc_data.iloc[0]["Recall"]
        status = "✓" if recall > 0.30 else "✗"
        print(f"  {status} Class {mc} recall: {recall:.4f} (target > 0.30)")